In [1]:
import sys
from pathlib import Path

root = Path().resolve()
sys.path.insert(0, str(root / "src"))


In [2]:
import tensorflow as tf

from FLRW_Net.network.network import NeuralNetwork

tf.keras.backend.set_floatx("float64")

In [3]:
flrw_net = NeuralNetwork(number_of_timesteps=1, triangulation="16-cell", cosmological_constant=1e-3)
adam_optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-3,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7,
    amsgrad=False,
    decay=0.9,
    clipnorm=1,
    clipvalue=None,
    global_clipnorm=None,
)
flrw_net.compile(optimizer=adam_optimizer)

In [4]:
import numpy as np
# inputs = tf.constant([[1, 2/5-3/8, 2, 2/5-3/8, 3]], dtype=tf.float64)  # 5-cell
inputs = tf.constant([[1, 1/2-3/8, 2]], dtype=tf.float64)  # 16-cell
# inputs = tf.constant([[1, (3 + np.sqrt(5)) / 2, 2]], dtype=tf.float64)  # 600-cell
flrw_net.training(inputs, epochs=10000)

Training:   0%|                           | 0/10000 [00:00<?, ?step/s]d:\Master Arbeit\PublishedOnGit\NeuralNetwork-FLRW\.venv\lib\site-packages\tensorflow\python\framework\indexed_slices.py:444: UserWarning: Converting sparse IndexedSlices(IndexedSlices(indices=Tensor("gradients/gradients/gradients/GatherV2_grad/UnsortedSegmentSum_grad/GatherV2_grad/Reshape_1:0", shape=(3,), dtype=int32), values=Tensor("gradients/gradients/gradients/GatherV2_grad/UnsortedSegmentSum_grad/GatherV2_grad/Reshape:0", shape=(3, None), dtype=float64), dense_shape=Tensor("gradients/gradients/gradients/GatherV2_grad/UnsortedSegmentSum_grad/GatherV2_grad/Cast:0", shape=(2,), dtype=int32))) to a dense Tensor of unknown shape. This may consume a large amount of memory.
  warnings.warn(
Training: 100%|██████████████| 10000/10000 [01:35<00:00, 104.40step/s]


(6.548831071572276e-29,
 [0.00019999999988488,
  2.6207652607588594,
  2.4802500331279966,
  1.8106918653896245,
  1.1990890473550793,
  0.7313374289138216,
  0.4001872366067075,
  0.18271085371469897,
  0.05674561875439206,
  0.00398150469030491,
  0.009920387996059637,
  0.040826931823650825,
  0.0684004271323946,
  0.08405167638267001,
  0.08735683355096141,
  0.0809193915830306,
  0.06811394515030271,
  0.05214588153721587,
  0.035711808401030005,
  0.020927995650851662,
  0.00937091366785348,
  0.0021573361092120267,
  3.126448006596592e-05,
  0.0016807214317521124,
  0.0039464161425875645,
  0.005405213354808587,
  0.005638107115824194,
  0.0047812714618021635,
  0.0032525269168919295,
  0.001587073049242352,
  0.0003414016389917205,
  3.9655202413834804e-05,
  0.0004756120182441344,
  0.0008693019983673889,
  0.0009324567524498613,
  0.0006758474578975931,
  0.0002804159485960324,
  1.2470197939845708e-05,
  0.00016916145437971533,
  0.0005120342198081833,
  0.000693691341731741

In [5]:
for key, value in flrw_net.model_params._asdict().items():
    print(key, value.numpy())
tf.print(flrw_net(inputs))

n1 24.0
n2 32.0
n3 16.0
nte 4.0
lamb 0.001
[[1 0.7071313364895393 2]]


In [6]:
from FLRW_Net.utils.losses import strut_losses, spatial_edge_losses

prediction = tf.constant([[1, 0.7071313364895393, 2]], dtype=tf.float64)
loss_struts = strut_losses(prediction, flrw_net.model_params)
loss_spatial_edges = spatial_edge_losses(prediction, flrw_net.model_params)
combined = tf.concat([loss_struts, loss_spatial_edges], axis=1)
loss = tf.squeeze(tf.reduce_mean(combined, axis=1))
tf.print(loss)

1.741011752565031e-24


In [ ]:
# Check whether the forward feed gives the same result
import tensorflow as tf

from src.OneStep.layer import HiddenLayer as Layer1
from src.TwoStep.layer import HiddenLayer as Layer2
from src.ThreeStep.layer import HiddenLayer as Layer3
from src.FourStep.layer import HiddenLayer as Layer4

layer1 = Layer1()
layer2 = Layer2()
layer3 = Layer3()
layer4 = Layer4()

inputs_1 = tf.constant([[1, 0.67, 2]], dtype=tf.float64)
inputs_2 = [[1, 0.67, 2, 0.68, 3]]
inputs_3 = tf.constant([[1, 0.67, 2, 0.68, 3, 0.69, 4]], dtype=tf.float64)
inputs_4 = tf.constant([[1, 0.67, 2, 0.68, 3, 0.69, 4, 0.7, 5]], dtype=tf.float64)

# print(layer1(inputs_1))
print(layer2(inputs_2))
# print(layer3(inputs_3))
# print(layer4(inputs_4))